[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SheffieldML/notebook/blob/main/GPc/gplvm_oil100.ipynb)

# MAP GPLVM on oil (100) with `gpc`

GPc port of the oil-flow GPLVM demo ([gplvm_oil100](https://github.com/SheffieldML/GPc/blob/main/docs/tutorials/gplvm_oil100.md); GPy `gplvm_oil_100` / `GPy.models.GPLVM`).

A GPLVM is an **unsupervised** GP: high-dimensional observations \(Y\) are treated as the GP outputs, and a low-dimensional latent \(\mathbf{X}\) is optimised (here a MAP point estimate). Class labels in the oil data colour the scatter only — they are not supervised targets.

```python
m = gpc.models.GPLVM(Y, input_dim=2)
m.optimize()
Xlat = m.latent_mean()
```

**Notes**

* Requires `gpc>=0.1.0a10` (also on `0.1.0a7`).
* `objective()` is **−log-likelihood**.
* Bayesian / variational sibling: [bgplvm_oil100](./bgplvm_oil100.ipynb) (`q(\mathbf{X})`, inducing inputs, RBF-ARD).
* pods stores features in `data["X"]` and one-hot (±1) labels in `data["Y"]` — we treat features as the GPLVM observations.


In [ ]:
import importlib
import re
import subprocess
import sys


def _gpc_ver_tuple(v: str):
    """Parse ``0.1.0a10`` / ``0.1.0`` into a comparable tuple (final > any alpha)."""
    m = re.fullmatch(r"(\d+)\.(\d+)\.(\d+)(?:a(\d+))?", str(v))
    if not m:
        return (0, 0, 0, 0)
    maj, mir, pat, alpha = m.groups()
    pre = int(alpha) if alpha is not None else 10**9
    return (int(maj), int(mir), int(pat), pre)


_MIN = "0.1.0a10"
_need = _gpc_ver_tuple(_MIN)


def _ensure_gpc():
    try:
        import gpc as _gpc
        if _gpc_ver_tuple(getattr(_gpc, "__version__", "0")) >= _need:
            return _gpc
    except ImportError:
        pass
    # Prefer TestPyPI for pre-release alphas; real PyPI for dependencies (CIP-002C).
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "--upgrade",
            "--no-cache-dir",
            "--index-url",
            "https://test.pypi.org/simple/",
            "--extra-index-url",
            "https://pypi.org/simple/",
            f"gpc>={_MIN}",
        ]
    )
    if "gpc" in sys.modules:
        importlib.reload(sys.modules["gpc"])
    import gpc as _gpc
    if _gpc_ver_tuple(getattr(_gpc, "__version__", "0")) < _need:
        raise RuntimeError(
            f"need gpc>={_MIN}, got {getattr(_gpc, '__version__', '?')}"
        )
    return _gpc


gpc = _ensure_gpc()


In [ ]:
import os
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")

import numpy as np
import gpc
from matplotlib import pyplot as plt

try:
    import pods
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pods"])
    import pods

try:
    import matplotlib
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "matplotlib"])


## Load oil_100

One hundred points from the three-phase oil-flow data (via `pods`). Each row of \(Y\) is a 12-D feature vector; the three flow regimes (non-flowing, slugging, homogeneous) are used only for colouring the latent plot. pods stores features in `data["X"]` and one-hot (±1) labels in `data["Y"]`.


In [ ]:
data = pods.datasets.oil_100()
Y = np.asarray(data["X"], dtype=float)  # pods: features in X, class indicators in Y
labels = np.asarray(data["Y"], dtype=float)
print("Y obs", Y.shape, "labels", labels.shape)
# one-hot ±1 → class index 0..2
cls = np.argmax(labels, axis=1)

## Fit MAP GPLVM

Optimise a 2-D latent \(\mathbf{X}\) and the GP hyperparameters jointly. The scatter of `latent_mean()` should separate the three regimes without ever seeing the labels during training — that is the usual oil-flow sanity check from the GPy / native demos.


In [ ]:
m = gpc.models.GPLVM(Y, input_dim=2)
print(m)
m.optimize(iters=80, messages=False)
print("objective", m.objective())
Xlat = m.latent_mean()
print("latent", Xlat.shape)

## Latent scatter coloured by oil class

Colour `latent_mean()` by regime. Separation without supervised labels is the usual oil-flow sanity check from the GPy / native demos.


In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
for c, name in enumerate(("non-flowing", "slugging", "homogeneous")):
    mask = cls == c
    ax.scatter(Xlat[mask, 0], Xlat[mask, 1], s=28, label=name, alpha=0.85)
ax.set_xlabel("latent 0")
ax.set_ylabel("latent 1")
ax.set_title("MAP GPLVM — oil_100")
ax.legend(fontsize=8)
fig.tight_layout()
plt.show()

## Reconstruct a few points

`predict` maps latent coordinates back to feature space. A small train RMSE on held-in points is a sanity check that the MAP embedding still explains \(Y\); the latent scatter is the teaching plot.


In [ ]:
mu, var = m.predict(Xlat[:5])
print("predict mu", mu.shape, "var", var.shape)
print("RMSE (train, first 5):", float(np.sqrt(np.mean((mu - Y[:5]) ** 2))))